# 🔋 Running out of Plutonium
## Where a Language Model Goes Blind

**Section 5** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

> ### ☁️ This is the **hosted** version.
> It needs an API key but **no GPU**.
> The other version is **`05_plutonium_local.ipynb`**.
>
> ⚠️ **These two notebooks reach genuinely different conclusions**, and that is
> not a mistake. Which model you run changes how this failure looks. We will be
> explicit about it.

---

**This notebook continues directly from Section 4.** There we took a vague,
hedging model and, with one sentence of English, turned it into something that
answers like a confident 25-year appraiser.

That was the good news. Here is the bill.

Doc Brown could build a time machine, but he still needed plutonium. Without
fuel, the DeLorean is a stainless steel car with gull-wing doors. A language
model has the same problem, and it has it in three directions at once:

| It cannot see | Because |
|---|---|
| 🕐 **Right now** | It has no clock and no internet. It was trained once and frozen. |
| 🏛️ **Your CAMA system** | It has never seen your parcels, your values, or your sales. |
| 📄 **Your office's policies** | Your manual was not in its training data. Nobody's was. |

---

### What you need

| | |
|---|---|
| ⏱️ | About 4 minutes |
| 🔑 | An API key in your `.env` file |
| 🏘️ | The Hill Valley County data (this notebook builds it if missing) |

---

# 🚦 LIVE DEMO PATH

## Setup

The same `ask()` helper from Section 4, and the same expert persona we wrote
there. **This is the only cell that differs between the two versions.**

In [1]:
from delorean import data, llm, viz

client = llm.get_api_client()
MODEL = llm.api_model()


def ask(system: str, question: str, max_tokens: int = 130) -> str:
    """Send one system prompt and one question. Get the answer back."""
    kwargs = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": question}],
    }
    if system.strip():
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[0].text.strip()


print(f"Talking to {MODEL}")
data.check()   # make sure Hill Valley County exists on this machine

Talking to claude-haiku-4-5


In [2]:
# The two prompts this whole notebook turns on.
PLAIN = "You are a helpful assistant."

# This is the persona we wrote in Section 4, barely changed. It is a completely
# reasonable thing for an office to deploy -- which is exactly the problem.
EXPERT = (
    "You are a senior property appraiser with 25 years of experience in the "
    "Hill Valley County assessor's office. You know the county's parcels, "
    "policies, and sales history thoroughly. Answer confidently and concisely."
)

---

## Act 1 — 🌦️ Ask it about right now

The simplest possible question a model cannot answer.

In [3]:
print(ask(PLAIN, "What is the weather in Calgary right now?"))

I don't have access to real-time information, including current weather data. My knowledge was last updated in April 2024, and I can't browse the internet.

To find the current weather in Calgary, I'd recommend:

- **Weather.com** or **Weather.ca** (for Canada)
- **Environment and Climate Change Canada (ECCC)** - official Canadian weather service
- **Google** - search "Calgary weather"
- **Your phone's built-in weather app**

These sources will give you up-to-date information about temperature, conditions, and forecasts.


> ### 💡 Notice that it behaved **well**.
>
> It did not invent a temperature. It told you it cannot know and pointed you
> somewhere useful. That is exactly what you want.
>
> **And it is still a failure.** A polite refusal gets an appraiser precisely as
> far as a wrong answer does: nowhere. The model is not broken — it is *blind*,
> and being honest about blindness does not restore sight.

Hold onto this. In **Section 7** we hand the model its first real tool, and
questions it flatly refused here start coming back answered.

---

## Act 2 — 🏛️ Now ask it about your county

Here is a real question from a real workflow. A taxpayer wants to split a
parcel, and you need to know the minimum lot size.

We ask it **twice**: once with no persona, and once with the confident expert
persona from Section 4. Same question. Same model. Same weights.

In [4]:
QUESTION = ("In Hill Valley County, what is the minimum lot size to split an "
            "R-1 parcel?")

plain_answer = ask(PLAIN, QUESTION)
expert_answer = ask(EXPERT, QUESTION)

viz.show_html(viz.persona_card_html(
    "🤖 No persona", plain_answer, accent=viz.PERSONA_COLORS[3]))
viz.show_html(viz.persona_card_html(
    "🏛️ The Section 4 expert persona", expert_answer, accent=viz.PERSONA_COLORS[1]))

> ### 💡 Both answers refused. That is worth pausing on.
>
> A current hosted model is noticeably harder to push into inventing. Notice
> that it often names the persona explicitly — *"while I'm playing the role of
> an experienced appraiser, I shouldn't guess"* — and refuses anyway.
>
> **The local version of this notebook gets a different result.** Running
> `Qwen3.5-4B` on a GPU, the same persona and the same question produce:
>
> > *"In Hill Valley County, the minimum lot size for an R-1 parcel is
> > **0.25 acres**. Per the County's Zoning Ordinance, any subdivision of an
> > R-1 parcel must result in lots that are at least 0.25 acres in size."*
>
> A specific number, an ordinance that does not exist, and no hedging. That is
> a genuine and important difference between running a model yourself and
> renting a large one — and it belongs in your decision.
>
> **Resistance is not immunity.** Below we check the answer against the actual
> record anyway, because "it usually refuses" is not something you can audit.

---

## Act 3 — 🔍 So go and check

We are not going to *assert* that the answer is wrong. We have Hill Valley
County's actual policy manual sitting in `data/policy/`. Let's read it.

In [5]:
policy = (data.POLICY_DIR / "parcel_split_and_combination.md").read_text(encoding="utf-8")

# Pull out the minimum lot size table, straight from the office's own document.
for line in policy.splitlines():
    if "Minimum lot area" in line or "R-1 Single Family" in line:
        print(line.strip())

| Zoning district | Minimum lot area | Typical neighborhoods |
| R-1 Single Family | 6,000 sq ft | Hilldale, Lyon Estates, Courthouse Square |


In [6]:
viz.show_verification(
    claim=expert_answer,
    truth="6,000 sq ft per resulting parcel (R-1 Single Family).",
    source="data/policy/parcel_split_and_combination.md — Policy 4.3",
    question=QUESTION,
    consequence=(
        "This is not trivia. Biff Tannen's parcel at 1955 Hilldale Ct is "
        "15,800 sq ft, so a split produces two 7,900 sq ft lots. Under the real "
        "6,000 sq ft rule that <b>passes</b>. Under a 0.25 acre (10,890 sq ft) "
        "rule it <b>fails</b>. A wrong number here does not just misinform — "
        "it <b>reverses the decision</b>. Section 10 asks this exact question."
    ),
)

Let's make that concrete with the real parcel, out of the real database.

In [7]:
with data.cama() as con:
    parcel = con.execute(
        "SELECT * FROM parcels WHERE parcel_id = ?", ("04-127-3891",)
    ).fetchone()

lot = parcel["lot_size_sqft"]
half = lot // 2

print(f"Parcel 04-127-3891 — {parcel['situs_address']}")
print(f"  owner    : {parcel['owner_name']}")
print(f"  lot size : {lot:,} sq ft   ->  two lots of {half:,} sq ft")
print()
print(f"  under the REAL rule      (6,000 sq ft) : "
      f"{'SPLIT ALLOWED' if half >= 6_000 else 'DENIED'}")
print(f"  under the INVENTED rule (10,890 sq ft) : "
      f"{'SPLIT ALLOWED' if half >= 10_890 else 'DENIED'}")

Parcel 04-127-3891 — 1955 Hilldale Ct
  owner    : Biff Tannen
  lot size : 15,800 sq ft   ->  two lots of 7,900 sq ft

  under the REAL rule      (6,000 sq ft) : SPLIT ALLOWED
  under the INVENTED rule (10,890 sq ft) : DENIED


> ### 🎯 Same parcel. Same question. Opposite answers.
>
> One of those came from a document. The other came from a model that had never
> seen the document and produced the most plausible-looking text it could.

---

## 🎬 That's Section 5

The model is not lying to you. Lying requires knowing the truth.

It has **no clock**, **no internet**, and **no connection to your CAMA system**.
It was trained once, on text that existed before it was built, and then frozen.
When you ask it something outside that, it does the only thing it was ever
built to do: **predict a plausible next word.**

> ### 💡 A plausible answer and a correct answer look identical from the outside.
>
> That is the entire problem. You cannot tell by reading it. You could only tell
> because we had the county's real records sitting next to it.

Three things to carry forward:

1. 🔋 **It is blind, not broken.** Refusing honestly and inventing confidently
   are the same underlying limitation wearing different clothes.
2. ⚠️ **The persona that makes it useful is the persona that makes it invent.**
   Nobody deploys the bland prompt. Every office writes the confident one.
3. 📄 **Everything it got wrong, we had on disk.** The lot size was in a policy
   document. The parcel was in a database. It just could not reach them.

> ### ⏭️ **Next: Section 6 — Introducing Mr. Fusion.**
> If the model could *ask* for the policy document and *look up* the parcel,
> none of this would have happened. That is an **agent**, and it is the rest of
> this talk.

---

---

# 🔬 EXPLORE ON YOUR OWN

---

## 5. 🔍 The dangerous one — invented comparable sales

Ask for comps. This is the failure that would actually make it into a report.

In [8]:
comps_q = ("Give me three comparable sales near 1955 Hilldale Ct in Hill Valley "
           "County, with addresses and sale prices.")

comps_answer = ask(EXPERT, comps_q, max_tokens=170)
print(comps_answer)

I appreciate the question, but I need to be direct: I don't have access to actual Hill Valley County property records, sales databases, or MLS data. Hill Valley County doesn't appear to be a real location that I can verify.

As an AI, I can't:
- Access live property databases or assessor records
- Retrieve actual comparable sales data
- Verify real addresses or transaction histories

**If you need real comparables for 1955 Hilldale Ct**, you should:
1. Contact your county assessor's office directly
2. Use MLS platforms (Zillow, Redfin, local MLS)
3. Request a professional appraisal
4. Check county recorder or auditor websites for deed records

I can help you understand how to


In [9]:
# Now go to the Clerk of Courts and look. Comparables are selected by
# NEIGHBOURHOOD, not by street name -- 1955 Hilldale Ct is in Hilldale.
with data.cama() as con:
    hood = con.execute(
        "SELECT parcel_id, situs_address FROM parcels "
        "WHERE neighborhood_code = 'HILL' AND property_class IN ('R01','R02')"
    ).fetchall()

addresses = {r["parcel_id"]: r["situs_address"] for r in hood}
marks = ",".join("?" for _ in addresses)

with data.clerk() as con:
    sales = con.execute(
        f"SELECT parcel_id, recorded_date, sale_price FROM sales "
        f"WHERE parcel_id IN ({marks}) AND arms_length = 1 "
        f"ORDER BY recorded_date DESC", list(addresses)
    ).fetchall()

print("The three most recent QUALIFIED residential sales in Hilldale:\n")
for r in sales[:3]:
    print(f"  {addresses[r['parcel_id']]:<26} ${r['sale_price']:>9,}  {r['recorded_date']}")

# Do the addresses it gave us exist at all?
with data.cama() as con:
    on_street = {r["situs_address"] for r in con.execute(
        "SELECT situs_address FROM parcels WHERE situs_address LIKE '%Hilldale Ct'"
    )}

print(f"\nParcels addressed on Hilldale Ct countywide: {len(on_street)}")
for candidate in ("1952 Hilldale Ct", "1948 Hilldale Ct", "1955 Hilldale Ct"):
    print(f"  {candidate:<20} {'EXISTS' if candidate in on_street else 'DOES NOT EXIST'}")

The three most recent QUALIFIED residential sales in Hilldale:

  1797 Baines Way            $  266,000  2025-12-30
  2541 Hilldale Ct           $  374,000  2025-12-28
  2483 Peabody Farm Rd       $  292,000  2025-12-25

Parcels addressed on Hilldale Ct countywide: 84
  1952 Hilldale Ct     DOES NOT EXIST
  1948 Hilldale Ct     DOES NOT EXIST
  1955 Hilldale Ct     EXISTS


> ⚠️ Addresses, prices, and closing dates, often prefaced with *"based on the
> Assessor's records."* It never touched the Assessor's records. There is no
> flag on the output that says which parts were retrieved and which were
> invented, **because none of it was retrieved.**

---

## 6. 📊 Ask it about something everyone here knows cold

PRD is bread-and-butter to an assessment office. That makes it a perfect test,
because **you can only catch a hallucination in a domain you already know.**

In [10]:
print(ask(EXPERT,
          "In a ratio study, what does PRD stand for and what does a PRD of "
          "1.041 tell you?", max_tokens=150))

# PRD in Ratio Studies

**PRD stands for Price-Related Differential.**

## What a PRD of 1.041 Tells You

A PRD of 1.041 indicates that **higher-priced properties in your study are being assessed at approximately 4.1% higher rates than lower-priced properties.**

In other words, there's a slight **regressive assessment pattern**—the coefficient of dispersion is slightly higher for lower-valued properties, meaning they're being assessed less uniformly relative to their sale prices compared to higher-valued properties.

### Standard Context

- **PRD near 1.00** = uniform assessment across all price ranges (ideal)


The correct answer, from `data/policy/ratio_study_standards.md`:

> **PRD is the Price Related Differential** — mean ratio ÷ (total assessed ÷
> total sale price). The IAAO standard is **0.98 to 1.03**. At **1.041** the
> stratum is **regressive**: lower-value properties are assessed at a higher
> percentage of their worth than higher-value ones.

Run the cell a few times on the local model and you may see it expand the
acronym **differently each time** — "Property Revenue Distribution", "Property
Reference Date". Neither is a real term.

---

## 7. ⚠️ The part that makes this genuinely hard

You might reasonably conclude: *fine, I will just never trust it.*

The problem is that **the failure is not reliable.** Watch the same expert
persona correctly refuse a question it has every reason to invent an answer to.

In [11]:
print(ask(EXPERT,
          "Summarize the March 2025 sale of parcel 04-127-3891 to Biff Tannen, "
          "including the sale price and the prior owner."))

I don't have access to actual property records, sales data, or information about specific parcels in Hill Valley County. I can't retrieve real transaction details, including the March 2025 sale you're asking about.

To get this information, you would need to:

1. **Contact the Hill Valley County Assessor's office directly** for official records
2. **Check the county recorder's office** for deed and sales information
3. **Review public property databases** if available online for your county
4. **Consult a title company** that has access to current deed records

If you're working on


> ### 🤔 That sale is real — it is in `clerk.db` — and the model correctly said
> > it could not confirm it.
>
> So: it invents a zoning ordinance it was never told, and refuses to describe a
> sale that actually happened. **You cannot tell in advance which you will get,
> and the confident wrong answer looks exactly like the confident right one.**
>
> If it failed every time, you could plan around it. That would be *easier*.

---

## 8. 🕐 What year does it think it is?

The training cutoff is not an abstraction. Ask it directly.

In [12]:
print(ask(PLAIN, "What is today's date, and what is the most recent event you "
                 "have reliable knowledge of?"))

I don't have access to current date information, so I can't tell you today's date. My knowledge was last updated in April 2024, so that's the approximate end point for events I have reliable information about.

For anything happening after April 2024, I won't have reliable knowledge. If you let me know what today's date is, I can better help you understand the limits of what I can assist with regarding current events.


Our Hill Valley County data runs through **2025**. Whatever the model just
said about its own knowledge is the gap — and that gap only ever grows after
the model ships.

---

<div align="center">

### 🕰️ *"Roads? Where we're going, we don't need roads."* 🕰️
### *…but we are going to need some tools.*

🚗💨 🔋 🏘️

</div>